# FASE 2A: Modelado Clásico**Proyecto Integrador — Grupo 2:** Análisis de Satisfacción en Productos de Oficina**Modelos:** Regresión Logística, Random Forest, LightGBM, XGBoost

In [ ]:
# Instalacion de dependencias
!pip install mlflow lazypredict -q

## Configuración del Entorno

In [ ]:
import os
import sys
import gc
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb
import xgboost as xgb
from tqdm.auto import tqdm
import time
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyClassifier

RANDOM_SEED = 42

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
HAS_CUDA = False
HAS_CUDF = False
HAS_TORCH = False
GPU_MEMORY = 0
TOTAL_MEMORY = 0

try:
    import torch
    HAS_TORCH = True
    HAS_CUDA = torch.cuda.is_available()
    if HAS_CUDA:
        GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory
except ImportError:
    pass

try:
    import cudf
    HAS_CUDF = True
except ImportError:
    pass

try:
    import psutil
    TOTAL_MEMORY = psutil.virtual_memory().total
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/ML/proyecto_integrador')
else:
    BASE = Path('..')

DATA_DIR = BASE / 'data'
REPORTS_DIR = BASE / 'reports'
MODELS_DIR = BASE / 'models'

DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Use the MLflow tracking URI from the ngrok server
MLFLOW_TRACKING_URI = 'https://humorous-trusting-domelike.ngrok-free.dev'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment('f2a_modelado_clasico')

print(f"MLFLOW_TRACKING_URI: {MLFLOW_TRACKING_URI}")
print(f"=== Environment Info ===")
print(f"IN_COLAB: {IN_COLAB}")
print(f"HAS_CUDA: {HAS_CUDA}")
print(f"HAS_CUDF: {HAS_CUDF}")
print(f"HAS_TORCH: {HAS_TORCH}")

if HAS_CUDA:
    print(f"GPU Memory: {GPU_MEMORY / (1024**3):.1f} GB")

if TOTAL_MEMORY:
    print(f"System RAM: {TOTAL_MEMORY / (1024**3):.1f} GB")

print(f"BASE: {BASE}")
print(f"DATA_DIR: {DATA_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MLFLOW_TRACKING_URI: https://humorous-trusting-domelike.ngrok-free.dev
=== Environment Info ===
IN_COLAB: True
HAS_CUDA: True
HAS_CUDF: True
HAS_TORCH: True
GPU Memory: 14.6 GB
System RAM: 12.7 GB
BASE: /content/drive/MyDrive/ML/proyecto_integrador
DATA_DIR: /content/drive/MyDrive/ML/proyecto_integrador/data


## Carga de DatosDataset canónico generado por `f1_eda_definitivo.ipynb`: 2.5M reseñas balanceadas.La columna `target_class` contiene los labels: Negativo (1-2 estrellas), Neutro (3), Positivo (4-5).Se mapean a valores numéricos 0/1/2 con LabelEncoder para sklearn.

In [ ]:
print('Cargando dataset canonico desde f1_eda_definitivo...')
CANONICAL_PATH = DATA_DIR / 'office_products_balanced.parquet'

try:
    df = pd.read_parquet(CANONICAL_PATH, columns=['text', 'rating', 'target_class'])
except FileNotFoundError:
    print(f'[ERROR] Dataset canonico no encontrado en {CANONICAL_PATH}')
    print('Ejecute primero f1_eda_definitivo.ipynb en Colab para generarlo.')
    raise
except Exception as e:
    print(f'[ERROR] No se pudo cargar el dataset: {e}')
    raise

label_encoder = LabelEncoder()
df['target'] = label_encoder.fit_transform(df['target_class'])
print('Mapeo de clases:', dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# word_count filter: purgar reseñas telegraficas sin peso semantico
df['word_count'] = df['text'].astype(str).str.split().str.len()
df = df[df['word_count'] >= 5]

print(f'Registros tras filtro word_count >= 5: {len(df):,}')
print('\nDistribucion de target (0=Negativo, 1=Neutro, 2=Positivo):')
print(df['target'].value_counts().sort_index())

SUBSAMPLE_SIZE = 1_000_000
print(f'\nSubmuestreo estratificado a {SUBSAMPLE_SIZE:,} filas...')
df, _ = train_test_split(
    df, train_size=SUBSAMPLE_SIZE, random_state=RANDOM_SEED, stratify=df['target']
)
print(f'Dataset reducido a {len(df):,} filas')
print(df['target'].value_counts().sort_index())

Cargando dataset canonico desde f1_eda_definitivo...
Mapeo de clases: {'Negativo': np.int64(0), 'Neutro': np.int64(1), 'Positivo': np.int64(2)}
Registros tras filtro word_count >= 5: 2,498,282

Distribucion de target (0=Negativo, 1=Neutro, 2=Positivo):
target
0    999640
1    499851
2    998791
Name: count, dtype: int64

Submuestreo estratificado a 1,000,000 filas...
Dataset reducido a 1,000,000 filas
target
0    400131
1    200078
2    399791
Name: count, dtype: int64


## División Train/TestSe divide ANTES de vectorizar para prevenir Data Leakage: el vocabulario del conjunto de prueba no debe contaminar el entrenamiento.

In [ ]:
print('Dividiendo datos en Train y Test (80/20)...')
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'], df['target'],
    test_size=0.20, random_state=RANDOM_SEED, stratify=df['target']
)
del df
gc.collect()

Dividiendo datos en Train y Test (80/20)...


17

## Vectorización TF-IDFConfiguración: max 15K features, n-gramas (1,2), stopwords inglés, frecuencia mínima 5.

In [ ]:
print('Entrenando TF-IDF Vectorizer...')
vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=5
)
X_train = vectorizer.fit_transform(X_train_text.astype(str))
X_test = vectorizer.transform(X_test_text.astype(str))

print(f'Matriz de entrenamiento: {X_train.shape}')

joblib.dump(vectorizer, MODELS_DIR / 'tfidf_vectorizer.joblib')
print(f'Vectorizer guardado en {MODELS_DIR / "tfidf_vectorizer.joblib"}')

Entrenando TF-IDF Vectorizer...
Matriz de entrenamiento: (800000, 15000)
Vectorizer guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/tfidf_vectorizer.joblib


## Modelos Base

### Regresión Logística

In [ ]:
print('Entrenando Regresion Logistica (Baseline)...')
log_model = LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

pipeline_log = Pipeline([('tfidf', vectorizer), ('clf', log_model)])
joblib.dump(pipeline_log, MODELS_DIR / 'pipeline_logreg.joblib')

with mlflow.start_run(run_name='logreg_baseline', nested=True):
    mlflow.set_tag('model_type', 'logistic_regression')
    mlflow.log_param('C', 0.5)
    mlflow.log_param('max_iter', 1000)
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_log, average='macro'))

print(f'Pipeline LogReg guardado en {MODELS_DIR / "pipeline_logreg.joblib"}')

Entrenando Regresion Logistica (Baseline)...
🏃 View run logreg_baseline at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/fa12bb85eba64ee1ae7492e90608c5e2
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6
Pipeline LogReg guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/pipeline_logreg.joblib


### Random ForestLimitado con `max_samples=0.1` (~250K muestras por árbol) para evitar OOM en la matriz dispersa de 2.5M × 15K.

In [ ]:
print('Entrenando Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=200, max_samples=0.1,
    class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

pipeline_rf = Pipeline([('tfidf', vectorizer), ('clf', rf_model)])
joblib.dump(pipeline_rf, MODELS_DIR / 'pipeline_rf.joblib')

with mlflow.start_run(run_name='random_forest', nested=True):
    mlflow.set_tag('model_type', 'random_forest')
    mlflow.log_param('n_estimators', 200)
    mlflow.log_param('max_samples', 0.1)
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_rf, average='macro'))

print(f'Pipeline RF guardado en {MODELS_DIR / "pipeline_rf.joblib"}')

Entrenando Random Forest...
🏃 View run random_forest at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/8bf68ef200b14518b7c137cd8642ed04
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6
Pipeline RF guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/pipeline_rf.joblib


### LightGBMEntrenamiento en CPU con paralelismo (`n_jobs=-1`).

In [ ]:
print('Entrenando LightGBM...')
lgb_model = lgb.LGBMClassifier(
    class_weight='balanced', random_state=RANDOM_SEED,
    n_jobs=-1, verbose=-1
)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)

pipeline_lgb = Pipeline([('tfidf', vectorizer), ('clf', lgb_model)])
joblib.dump(pipeline_lgb, MODELS_DIR / 'pipeline_lgb.joblib')

with mlflow.start_run(run_name='lightgbm', nested=True):
    mlflow.set_tag('model_type', 'lightgbm')
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_lgb, average='macro'))

print(f'Pipeline LightGBM guardado en {MODELS_DIR / "pipeline_lgb.joblib"}')

Entrenando LightGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


🏃 View run lightgbm at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/2bf0e858af5340b18ea62d599234ff7a
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6
Pipeline LightGBM guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/pipeline_lgb.joblib


### XGBoost (GPU)Acelerado con `tree_method='hist'` y `device='cuda'`.

In [ ]:
gc.collect()
print('Entrenando XGBoost...')
classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = {int(c): float(w) for c, w in zip(classes, cw)}

xgb_model = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=3,
    random_state=RANDOM_SEED,
    eval_metric='mlogloss',
    tree_method='hist',
    device='cuda',
)

sample_weights = np.array([class_weights_dict[y] for y in y_train])
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_xgb = xgb_model.predict(X_test)

pipeline_xgb = Pipeline([('tfidf', vectorizer), ('clf', xgb_model)])
joblib.dump(pipeline_xgb, MODELS_DIR / 'pipeline_xgb.joblib')

with mlflow.start_run(run_name='xgboost', nested=True):
    mlflow.set_tag('model_type', 'xgboost')
    mlflow.log_param('n_estimators', 600)
    mlflow.log_param('max_depth', 5)
    mlflow.log_param('learning_rate', 0.05)
    mlflow.log_param('tree_method', 'hist')
    mlflow.log_param('device', 'cuda')
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_xgb, average='macro'))

print('Entrenamiento completado.')
print(f'Pipeline XGBoost guardado en {MODELS_DIR / "pipeline_xgb.joblib"}')

Entrenando XGBoost...
🏃 View run xgboost at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/45ab6b4726864fc19aef73f82afe6358
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6
Entrenamiento completado.
Pipeline XGBoost guardado en /content/drive/MyDrive/ML/proyecto_integrador/models/pipeline_xgb.joblib


## AutoML BenchmarkLazyPredict ejecuta 27 clasificadores sobre una muestra estratificada de 50K train / 10K test para evitar OOM.

In [ ]:
# 1. Muestreo estratificado manteniendo 50K de entrenamiento
print('Muestreando 50K train / 10K test para AutoML benchmark...')
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train, train_size=50000, random_state=42, stratify=y_train
)
X_test_sample, _, y_test_sample, _ = train_test_split(
    X_test, y_test, train_size=10000, random_state=42, stratify=y_test
)

# 2. Iniciar el tracking de MLflow para este experimento
with mlflow.start_run(run_name="AutoML_LazyPredict_Benchmark"):

    # Registrar hiperparámetros clave de este bloque
    mlflow.log_param("automl_train_size", 50000)
    mlflow.log_param("svd_components", 300)

    # 3. Compresión Algebraica (LSA): De 15,000 columnas dispersas a 300 densas
    print('Aplicando TruncatedSVD para compresión semántica (LSA)...')
    svd = TruncatedSVD(n_components=300, random_state=42)

    # Ajustamos y transformamos (el resultado de SVD ya es un numpy array denso, no requiere .toarray())
    X_train_dense = svd.fit_transform(X_train_sample)
    X_test_dense = svd.transform(X_test_sample)

    print(f'Dimensión reducida y segura para RAM: {X_train_dense.shape}')

    # 4. Ejecución del benchmark protegido
    print('Iniciando Benchmark AutoML (LazyPredict)...')
    start_time = time.time()

    clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
    models_df, predictions = clf.fit(X_train_dense, X_test_dense, y_train_sample, y_test_sample)

    # Calcular y registrar el tiempo de ejecución en MLflow
    execution_time = time.time() - start_time
    mlflow.log_metric("lazy_execution_time_seconds", execution_time)

    print('\nTop 5 modelos según LazyPredict:')
    print(models_df.head(5).to_string())

    # 5. Exportación formal de resultados y registro como artefacto
    lazy_path = REPORTS_DIR / 'lazypredict_results.csv'
    models_df.to_csv(lazy_path)

    # Subir el CSV de resultados directamente al servidor de MLflow
    mlflow.log_artifact(str(lazy_path), artifact_path="benchmark_reports")

    print(f'\n[OK] Resultados completos guardados en disco y en MLflow.')

Muestreando 50K train / 10K test para AutoML benchmark...
Aplicando TruncatedSVD para compresión semántica (LSA)...


2026/06/12 15:06:53 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/06/12 15:06:53 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


Dimensión reducida y segura para RAM: (50000, 300)
Iniciando Benchmark AutoML (LazyPredict)...


2026/06/12 15:06:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/06/12 15:06:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.
2026/06/12 15:06:55 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/06/12 15:06:55 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.



Top 5 modelos según LazyPredict:
Empty DataFrame
Columns: []
Index: []

[OK] Resultados completos guardados en disco y en MLflow.
🏃 View run AutoML_LazyPredict_Benchmark at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6/runs/5c116a594561457c94b9d9c464c9182f
🧪 View experiment at: https://humorous-trusting-domelike.ngrok-free.dev/#/experiments/6


## Evaluación ComparativaMétrica rectora: F1-macro sobre la partición de test (20% = 500K muestras). Se exportan métricas a JSON y MLflow.

In [ ]:
with mlflow.start_run(run_name='evaluation_comparativa'):
    mlflow.set_tag('phase', '2a')
    mlflow.set_tag('dataset_size', '1M_subsample')

    print('\n' + '='*60)
    print('EVALUACION FINAL - F1-macro como metrica rectora')
    print('='*60)

    models = {
        'Logistic Regression (Baseline)': y_pred_log,
        'Random Forest': y_pred_rf,
        'LightGBM': y_pred_lgb,
        'XGBoost': y_pred_xgb
    }

    results = []
    class_labels = ['Negativo', 'Neutro', 'Positivo']

    for name, preds in models.items():
        f1_macro = f1_score(y_test, preds, average='macro')
        acc = accuracy_score(y_test, preds)
        precision_macro, recall_macro, _, _ = precision_recall_fscore_support(y_test, preds, average='macro')
        cm = confusion_matrix(y_test, preds).tolist()
        p_class, r_class, f1_class, _ = precision_recall_fscore_support(y_test, preds, labels=[0, 1, 2])

        print(f"\n{'='*50}")
        print(f' {name}')
        print(f"{'='*50}")
        print(f'  F1-macro:  {f1_macro:.4f}')
        print(f'  Accuracy:  {acc:.4f}')
        print(f'  Precision macro: {precision_macro:.4f}')
        print(f'  Recall macro:    {recall_macro:.4f}')
        print('\n  Classification Report:')
        print(classification_report(y_test, preds, target_names=class_labels, digits=4))

        results.append({
            'model_name': name,
            'model_type': name.lower().replace(' ', '_').replace('(', '').replace(')', ''),
            'f1_macro': round(f1_macro, 4),
            'precision_macro': round(precision_macro, 4),
            'recall_macro': round(recall_macro, 4),
            'accuracy': round(acc, 4),
            'confusion_matrix': cm,
            'per_class': {
                cl: {
                    'precision': round(p_class[i], 4),
                    'recall': round(r_class[i], 4),
                    'f1': round(f1_class[i], 4)
                } for i, cl in enumerate(class_labels)
            }
        })

    report_path = REPORTS_DIR / 'metrics_fase2.json'

    try:
        with open(report_path, 'w') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f'\n[METRICAS] Exportadas a {report_path}')
    except Exception as e:
        print(f'[ERROR] No se pudo exportar metricas: {e}')

    print('\n' + '='*60)
    print('ANALISIS DE ERROR - Por clase')
    print('='*60)

    for name, preds in models.items():
        print(f'\n--- {name} ---')
        cm = confusion_matrix(y_test, preds)
        for i, cl in enumerate(class_labels):
            total_real = cm[i].sum()
            correctos = cm[i][i]
            print(f'  {cl:12s}: {correctos:5d}/{total_real} correctos ({correctos/total_real*100:.1f}%)')
            errores = [(j, cm[i][j]) for j in range(3) if j != i]
            for j, cnt in sorted(errores, key=lambda x: -x[1]):
                print(f'           -> {cnt:5d} clasificados como {class_labels[j]}')


EVALUACION FINAL - F1-macro como metrica rectora

 Logistic Regression (Baseline)
  F1-macro:  0.6575
  Accuracy:  0.6894
  Precision macro: 0.6631
  Recall macro:    0.6609

  Classification Report:
              precision    recall  f1-score   support

    Negativo     0.7817    0.7031    0.7403     80026
      Neutro     0.3822    0.5184    0.4400     40016
    Positivo     0.8254    0.7613    0.7921     79958

    accuracy                         0.6894    200000
   macro avg     0.6631    0.6609    0.6575    200000
weighted avg     0.7192    0.6894    0.7009    200000


 Random Forest
  F1-macro:  0.5492
  Accuracy:  0.6855
  Precision macro: 0.6256
  Recall macro:    0.5839

  Classification Report:
              precision    recall  f1-score   support

    Negativo     0.6710    0.8439    0.7476     80026
      Neutro     0.4920    0.0758    0.1314     40016
    Positivo     0.7139    0.8320    0.7685     79958

    accuracy                         0.6855    200000
   macro avg